In [ ]:
# Run from the repository root so that `utils` and the relative data paths resolve
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Arial"
plt.rcParams['font.size'] = 14
import seaborn as sns
import numpy as np
import pandas as pd
import json
from collections import Counter, defaultdict
from pathlib import Path
from glob import glob

In [ ]:
def mae(run, gt=None, compute="raw"):

    # print(f"This is {compute} MAE calculation")

    if gt is None:
        gt = run
    
    # Load prediction data
    run_folder = Path(run) / "Test"
    pred_path_list = glob(str(run_folder / "*_norm.npy"))
    pred_arr_list = [np.load(path).T for path in sorted(pred_path_list)]   # (4, 719)
    pred_arr = np.concatenate(pred_arr_list, axis=1)  # (4, 17196)

    # Load ground truth data
    gt_folder = Path(gt) / "Test"
    gt_path_list = glob(str(gt_folder / "*_gt.npy"))
    gt_arr_list = [np.load(path).T for path in sorted(gt_path_list)]
    gt_arr = np.concatenate(gt_arr_list, axis=1)  # (4, 17196)

    # print(f"Pred shape: {pred_arr.shape}, GT shape: {gt_arr.shape}")


    # Calculate MAE for each class
    # mae_list = []
    # for i in range(4):
    #     mae = np.mean(np.abs(pred_arr[:, i] - gt_arr[:, i]))
    #     print(f"MAE for class {i}: {mae:.5f}")
    #     mae_list.append(mae)
        
    # Compute relative MAE
    # rmae_list = np.array(mae_list)
    # for i in range(4):
    #     rmae = np.abs(pred_arr[:, i] - gt_arr[:, i])
    #     gt_cls = gt_arr[:, i]
    #     rmae = np.mean(np.sum(rmae, axis=0) / np.sum(gt_cls, axis=0))
    #     print(f"Relative MAE for class {i}: {rmae:.5f}")

    # Minute aggregation
    pred_minute = pred_arr.reshape(4, -1, 12).sum(axis=2)
    gt_minute = gt_arr.reshape(4, -1, 12).sum(axis=2)

    # Quarter aggregation
    pred_qt = pred_arr.reshape(4, -1, 180).sum(axis=2)
    gt_qt = gt_arr.reshape(4, -1, 180).sum(axis=2)

    # Hourly aggregation
    pred_hour = pred_arr.reshape(4, -1, 720).sum(axis=2)
    gt_hour = gt_arr.reshape(4, -1, 720).sum(axis=2)

    compute_dict = {
        "raw": (pred_arr, gt_arr),
        "hour": (pred_hour, gt_hour),
        "quarter": (pred_qt, gt_qt),
        "minute": (pred_minute, gt_minute)
    }
    pred_to_compute, gt_to_compute = compute_dict[compute]  # (4, 17196) or (4, 24)
    # print(f"Pred shape: {pred_to_compute.shape}, GT shape: {gt_to_compute.shape}")
    
    mae = np.mean(np.abs(pred_to_compute - gt_to_compute), axis=1)
    print(f"MAE: {mae.round(2)}")

    # Compute mean gt
    mean_gt = np.mean(gt_to_compute, axis=1)
    # for i in range(4):
    #     print(f"Mean GT for class {i}: {mean_gt[i]:.5f}")
    
    rmae = 100* mae / mean_gt
    print(f"Relative MAE: {rmae.round(2)}")
    
    # rmae_detach = np.abs(pred_to_compute - gt_to_compute) / gt_to_compute
    # for i in range(4):
    #     rmae_cls = rmae_detach[i][np.isfinite(rmae_detach[i])]
    #     mean_rmae = np.mean(rmae_cls)
    #     print(f"Mean Relative MAE for class {i}: {100*mean_rmae:.5f}%")
        


# mae("runs/dual4", compute="minute")
# mae("runs/femlpacc", compute="hour")
# mae("runs/femlpstr", compute="hour")
# mae("runs/noaxle", gt="runs/noaxle", compute="hour")

In [ ]:
# EVALUATION accuracy
# Parametric Study
def accuracy(run, compute="raw", gt=None):

    if gt is None:
        gt = run

    # print(f"This is {compute} accuracy calculation")
    
    # Load prediction data
    run_folder = Path(run)  / "Test"
    pred_path_list = glob(str(run_folder / "*_norm.npy"))
    pred_arr_list = [np.load(path).T for path in sorted(pred_path_list)]   # (4, 719)
    pred_arr = np.concatenate(pred_arr_list, axis=1)  # (4, 17196)

    # Load ground truth data
    gt_folder = Path(gt)  / "Test"
    gt_path_list = glob(str(gt_folder / "*_gt.npy"))
    gt_arr_list = [np.load(path).T for path in sorted(gt_path_list)]
    gt_arr = np.concatenate(gt_arr_list, axis=1)  # (4, 17196)

    # Minute aggregation
    pred_minute = pred_arr.reshape(4, -1, 12).sum(axis=2)
    gt_minute = gt_arr.reshape(4, -1, 12).sum(axis=2)

    # Hourly aggregation
    pred_hour = pred_arr.reshape(4, -1, 720).sum(axis=2)
    gt_hour = gt_arr.reshape(4, -1, 720).sum(axis=2)

    compute_dict = {
        "raw": (pred_arr, gt_arr),
        "hour": (pred_hour, gt_hour),
        "minute": (pred_minute, gt_minute)
    }
    pred_to_compute, gt_to_compute = compute_dict[compute]  # (4, 17196) or (4, 24)

    # for b in [1, 5,10,20,30,60,600]:
    max_values = np.maximum(pred_to_compute, gt_to_compute)
    min_values = np.minimum(pred_to_compute, gt_to_compute)

    sum_max = np.sum(max_values, axis=1)
    sum_min = np.sum(min_values, axis=1)
    accuracy = 100*sum_min/sum_max
    print(f"Accuracy: {accuracy.round(2)}")
    
    # if plot:
    #     plt.figure(figsize=(8, 5))
    #     plt.plot(np.arange(9), output_squeeze[:, 3], label="Prediction")
    #     plt.plot(np.arange(59), gt_squeeze[:, 3], label="Ground Truth")
    #     plt.legend()
    #     # plt.savefig(f"figures/threshold_rounding.svg", format="svg", bbox_inches="tight")


In [ ]:

ttest = "runs/train14"
mae(ttest, compute="hour")
accuracy(ttest, compute="hour")

In [ ]:
print("Peak result")
mae("runs/peak_count", compute="hour")
accuracy("runs/peak_count", compute="hour")

print("\nDual result")
mae("runs/dual", compute="hour")
accuracy("runs/dual", compute="hour")

print("\nsingle acc reuslt")
mae("runs/singleacc", compute="hour")
accuracy("runs/singleacc", compute="hour")

print("\nsingle str reuslt")
mae("runs/singlestr", compute="hour")
accuracy("runs/singlestr", compute="hour")


In [ ]:
# EVALUATION accuracy
# Parametric Study
def mae_acc_2cls(run, gt=None):

    if gt is None:
        gt = run

    # print(f"This is {compute} accuracy calculation")
    
    # Load prediction data 
    run_folder = Path(run)  / "Test"
    pred_path_list = glob(str(run_folder / "*_norm.npy"))
    pred_arr_list = [np.load(path).T for path in sorted(pred_path_list)]   # (4, 719)
    pred_arr = np.concatenate(pred_arr_list, axis=1)  # (4, 17196)

    # Load ground truth data
    gt_folder = Path(gt)  / "Test"
    gt_path_list = glob(str(gt_folder / "*_gt.npy"))
    gt_arr_list = [np.load(path).T for path in sorted(gt_path_list)]
    gt_arr = np.concatenate(gt_arr_list, axis=1)  # (4, 17196)

    # Hourly aggregation
    pred_hour = pred_arr.reshape(4, -1, 720).sum(axis=2)
    gt_hour = gt_arr.reshape(4, -1, 720).sum(axis=2)


    pred_light = np.sum(pred_hour[[0,1],:], axis=0, keepdims=True)
    gt_light = np.sum(gt_hour[[0,1],:], axis=0, keepdims=True)

    pred_heavy = pred_hour[[3],:]
    gt_heavy = gt_hour[[3],:]

    pred_to_compute = np.concatenate([pred_light, pred_heavy], axis=0)  # (2, 24)
    gt_to_compute = np.concatenate([gt_light, gt_heavy], axis=0)  # (2, 24)

    mae = np.mean(np.abs(pred_to_compute - gt_to_compute), axis=1)
    

    # for b in [1, 5,10,20,30,60,600]:
    max_values = np.maximum(pred_to_compute, gt_to_compute)
    min_values = np.minimum(pred_to_compute, gt_to_compute)

    sum_max = np.sum(max_values, axis=1)
    sum_min = np.sum(min_values, axis=1)
    accuracy = 100*sum_min/sum_max


    # Compute mean gt
    mean_gt = np.mean(gt_to_compute, axis=1)

    
    rmae = 100* mae / mean_gt

    print(f"MAE: {mae.round(2)}, Relative MAE: {rmae.round(2)}, Accuracy: {accuracy.round(2)}")

    # return mae, accuracy
    
    # if plot:
    #     plt.figure(figsize=(8, 5))
    #     plt.plot(np.arange(9), output_squeeze[:, 3], label="Prediction")
    #     plt.plot(np.arange(59), gt_squeeze[:, 3], label="Ground Truth")
    #     plt.legend()
    #     # plt.savefig(f"figures/threshold_rounding.svg", format="svg", bbox_inches="tight")


print("Peak result")
mae_acc_2cls("runs/peak_count")


print("\nsingle reuslt")
mae_acc_2cls("runs/singleacc")
mae_acc_2cls("runs/singlestr")


print("\ncnn reuslt")
mae_acc_2cls("runs/cnnacc")
mae_acc_2cls("runs/cnnstr")

print("\nfegnn reuslt")
mae_acc_2cls("runs/fegnnacc")
mae_acc_2cls("runs/fegnnstr")

print("\nfemlp reuslt")
mae_acc_2cls("runs/femlpacc")
mae_acc_2cls("runs/femlpstr")


In [ ]:
def coeff_determination_numpy(y_pred, y_true):
    """
    Input shape is (4, 17196) or (4, 24)
    """
    # Calculate the mean of y_true for each target (per column)
    y_true_mean = np.mean(y_true, axis=1, keepdims=True)  # shape: (4,)  
    
    # Calculate the total sum of squares (variation of each target from its mean)
    total_sum_of_squares = np.sum((y_true - y_true_mean) ** 2, axis=1)  # shape: (3,)
    
    # Calculate the residual sum of squares (difference between predictions and actual values)
    residual_sum_of_squares = np.sum((y_true - y_pred) ** 2, axis=1)  # shape: (3,)

    # If total_sum_of_squares is 0, set R² to NaN 
    r2 = np.where(total_sum_of_squares != 0,
                  1 - (residual_sum_of_squares / total_sum_of_squares),
                  np.nan)  # You can also use -inf instead of NaN

    return r2

def eval_r2(run, compute="hour"):

    # print(f"This is {compute} accuracy calculation")
    
    # Load prediction data
    run_folder = Path(run)
    pred_path_list = glob(str(run_folder / "*_norm.npy"))
    pred_arr_list = [np.load(path).T for path in sorted(pred_path_list)]   # (4, 719)
    pred_arr = np.concatenate(pred_arr_list, axis=1)  # (4, 17196)

    # Load ground truth data
    gt_folder = Path(run)
    gt_path_list = glob(str(gt_folder / "*_gt.npy"))
    gt_arr_list = [np.load(path).T for path in sorted(gt_path_list)]
    gt_arr = np.concatenate(gt_arr_list, axis=1)  # (4, 17196)

    # Hourly aggregation
    pred_hour = pred_arr.reshape(4, -1, 720).sum(axis=2)
    gt_hour = gt_arr.reshape(4, -1, 720).sum(axis=2)

    # Minute aggregation
    pred_minute = pred_arr.reshape(4, -1, 12).sum(axis=2)
    gt_minute = gt_arr.reshape(4, -1, 12).sum(axis=2)

    compute_dict = {
        "raw": (pred_arr, gt_arr),
        "hour": (pred_hour, gt_hour),
        "minute": (pred_minute, gt_minute)
    }
    pred_to_compute, gt_to_compute = compute_dict[compute]  # (4, 17196) or (4, 24)

    r2 = coeff_determination_numpy(pred_to_compute, gt_to_compute)
    print(f"R2: {r2.round(2)}")

eval_r2("runs/train14/Test", "raw")
eval_r2("runs/train14/Test", "minute")
eval_r2("runs/train14/Test", "hour")

